In [ ]:
# Cell 1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/DocSnap_Project', exist_ok=True)
print("Google Drive mounted successfully")
print("Project folder: /content/drive/MyDrive/DocSnap_Project")

Mounted at /content/drive
Google Drive mounted successfully
Project folder: /content/drive/MyDrive/DocSnap_Project


In [ ]:
# Cell 2 — Install libraries and reload data
!pip install datasets transformers torch rouge-score nltk spacy scikit-learn sentencepiece -q
!python -m spacy download en_core_web_sm -q

import pandas as pd
import torch
import nltk
import re
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("GPU Available:", torch.cuda.is_available())
print("GPU Name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

# Download and prepare data
from datasets import load_dataset
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
import spacy

print("\nLoading CNN/DailyMail dataset...")
dataset = load_dataset("cnn_dailymail", "3.0.0")

# Quick preprocessing of 5000 articles
sample_data = dataset["train"].select(range(5000))
articles = [x["article"] for x in sample_data]
highlights = [x["highlights"] for x in sample_data]

df = pd.DataFrame({'article': articles, 'highlights': highlights})
df['article_word_count'] = df['article'].apply(lambda x: len(x.split()))

# Save to Drive permanently
df.to_csv('/content/drive/MyDrive/DocSnap_Project/docsnap_data.csv', index=False)

print(f"Data ready — {len(df)} articles")
print(f"Average article length: {df['article_word_count'].mean():.0f} words")
print("Data saved to Google Drive permanently")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 94.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
GPU Available: True
GPU Name: Tesla T4

Loading CNN/DailyMail dataset...


README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

...

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Data ready — 5000 articles
Average article length: 615 words
Data saved to Google Drive permanently


In [ ]:
# Cell 3 — TextRank Extractive Summarization
import numpy as np
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings
warnings.filterwarnings('ignore')

print("=== TEXTRANK EXTRACTIVE SUMMARIZATION ===\n")

def textrank_summarize(text, num_sentences=3):
    """
    TextRank Algorithm:
    1. Split text into sentences
    2. Build TF-IDF vectors for each sentence
    3. Calculate similarity between all sentence pairs
    4. Build similarity graph
    5. Rank sentences using graph scores
    6. Return top N sentences as summary
    """
    # Step 1 — Split into sentences
    sentences = sent_tokenize(text)

    if len(sentences) < 3:
        return text

    # Step 2 — TF-IDF vectors
    vectorizer = TfidfVectorizer()
    try:
        tfidf_matrix = vectorizer.fit_transform(sentences)
    except:
        return ' '.join(sentences[:num_sentences])

    # Step 3 — Cosine similarity matrix
    similarity_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

    # Step 4 — Graph scoring (PageRank-style)
    scores = np.sum(similarity_matrix, axis=1)

    # Step 5 — Rank sentences
    ranked_indices = np.argsort(scores)[::-1][:num_sentences]
    ranked_indices = sorted(ranked_indices)

    # Step 6 — Return top sentences
    summary = ' '.join([sentences[i] for i in ranked_indices])
    return summary

# Test on 5 real articles
print("Testing TextRank on 5 real articles:\n")
print("=" * 60)

results = []
for i in range(5):
    article = df['article'][i]
    reference = df['highlights'][i]
    summary = textrank_summarize(article, num_sentences=3)

    results.append({
        'article_length': len(article.split()),
        'summary_length': len(summary.split()),
        'reference': reference,
        'summary': summary
    })

    print(f"Article {i+1} ({len(article.split())} words):")
    print(f"TextRank Summary: {summary[:200]}...")
    print(f"Reference Summary: {reference[:150]}...")
    print(f"Compression: {len(article.split())} → {len(summary.split())} words")
    print("-" * 60)

print("\nTextRank implementation successful")
print(f"Average input length: {np.mean([r['article_length'] for r in results]):.0f} words")
print(f"Average summary length: {np.mean([r['summary_length'] for r in results]):.0f} words")

=== TEXTRANK EXTRACTIVE SUMMARIZATION ===

Testing TextRank on 5 real articles:

Article 1 (455 words):
TextRank Summary: Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash aw...
Reference Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's e...
Compression: 455 → 97 words
------------------------------------------------------------
Article 2 (698 words):
TextRank Summary: So, he says, the sheer volume is overwhelming the system, and the result is what we see on the ninth floor. Of course, it is a jail, so it's not supposed to be warm and comforting, but the lights glar...
Reference Summary: Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While

In [ ]:
# Cell 4 — BERT Extractive Summarization
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

print("=== BERT EXTRACTIVE SUMMARIZATION ===\n")
print("Loading BERT model... (this takes 2-3 minutes)")

# Load pre-trained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model = model.cuda()  # Move to GPU
model.eval()

print("BERT model loaded successfully")
print(f"Model parameters: ~110 million")

def get_bert_embeddings(sentences):
    """Get BERT embeddings for each sentence"""
    embeddings = []
    for sent in sentences:
        # Tokenize
        inputs = tokenizer(
            sent,
            return_tensors='pt',
            max_length=128,
            truncation=True,
            padding=True
        ).to('cuda')

        # Get embeddings
        with torch.no_grad():
            outputs = model(**inputs)

        # Use [CLS] token as sentence embedding
        embedding = outputs.last_hidden_state[:, 0, :].cpu().numpy()
        embeddings.append(embedding[0])

    return np.array(embeddings)

def bert_summarize(text, num_sentences=3):
    """
    BERT Extractive Summarization:
    1. Split into sentences
    2. Get BERT embeddings for each sentence
    3. Calculate cosine similarity
    4. Rank and select top sentences
    """
    sentences = sent_tokenize(text)

    if len(sentences) < 3:
        return text

    # Limit to first 20 sentences for speed
    sentences = sentences[:20]

    # Get BERT embeddings
    embeddings = get_bert_embeddings(sentences)

    # Calculate similarity matrix
    similarity_matrix = cosine_similarity(embeddings)

    # Score sentences
    scores = np.sum(similarity_matrix, axis=1)

    # Get top sentences
    ranked_indices = np.argsort(scores)[::-1][:num_sentences]
    ranked_indices = sorted(ranked_indices)

    summary = ' '.join([sentences[i] for i in ranked_indices])
    return summary

# Test on 5 articles
print("\nTesting BERT on 5 real articles:\n")
print("=" * 60)

bert_results = []
for i in range(5):
    article = df['article'][i]
    reference = df['highlights'][i]
    summary = bert_summarize(article, num_sentences=3)

    bert_results.append({
        'article_length': len(article.split()),
        'summary_length': len(summary.split()),
        'summary': summary,
        'reference': reference
    })

    print(f"Article {i+1} ({len(article.split())} words):")
    print(f"BERT Summary: {summary[:200]}...")
    print(f"Reference:    {reference[:150]}...")
    print(f"Compression: {len(article.split())} → {len(summary.split())} words")
    print("-" * 60)

print("\nBERT extractive summarization successful")
print(f"Average input length: {np.mean([r['article_length'] for r in bert_results]):.0f} words")
print(f"Average output length: {np.mean([r['summary_length'] for r in bert_results]):.0f} words")

=== BERT EXTRACTIVE SUMMARIZATION ===

Loading BERT model... (this takes 2-3 minutes)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded successfully
Model parameters: ~110 million

Testing BERT on 5 real articles:

Article 1 (455 words):
BERT Summary: Details of how he'll mark his landmark birthday are under wraps. His agent and publicist had no comment on his plans. Despite his growing fame and riches, the actor says he is keeping his feet firmly ...
Reference:    Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's e...
Compression: 455 → 39 words
------------------------------------------------------------
Article 2 (698 words):
BERT Summary: The prisoners are wearing sleeveless robes. They're designed to keep the mentally ill patients from injuring themselves. That's also why they have no shoes, laces or mattresses....
Reference:    Mentally ill inmates in Miami are housed on the "forgotten floor"
Judge Steven Leifman says most are there as a result of "avoidable felonies"
While C...
Compression: 698 → 

In [ ]:
# Cell 5 — T5 Abstractive Summarization
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch
import warnings
warnings.filterwarnings('ignore')

print("=== T5 ABSTRACTIVE SUMMARIZATION ===\n")
print("Loading T5 model... (this takes 3-4 minutes)")

# Load T5
t5_tokenizer = T5Tokenizer.from_pretrained('t5-small')
t5_model = T5ForConditionalGeneration.from_pretrained('t5-small')
t5_model = t5_model.cuda()
t5_model.eval()

print("T5 model loaded successfully")
print("Model: t5-small (~60 million parameters)")

def t5_summarize(text, max_length=150, min_length=40):
    """
    T5 Abstractive Summarization:
    1. Prepend 'summarize:' prefix (T5 requirement)
    2. Tokenize input
    3. Generate summary tokens
    4. Decode to text
    """
    # T5 requires 'summarize: ' prefix
    input_text = "summarize: " + text

    # Tokenize
    inputs = t5_tokenizer(
        input_text,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding=True
    ).to('cuda')

    # Generate summary
    with torch.no_grad():
        summary_ids = t5_model.generate(
            inputs['input_ids'],
            max_length=max_length,
            min_length=min_length,
            length_penalty=2.0,
            num_beams=4,
            early_stopping=True
        )

    # Decode
    summary = t5_tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )
    return summary

# Test on 5 real articles
print("\nTesting T5 on 5 real articles:\n")
print("=" * 60)

t5_results = []
for i in range(5):
    article = df['article'][i]
    reference = df['highlights'][i]

    # Use first 800 words to stay within T5 limits
    article_truncated = ' '.join(article.split()[:800])

    summary = t5_summarize(article_truncated)

    t5_results.append({
        'article_length': len(article.split()),
        'summary_length': len(summary.split()),
        'summary': summary,
        'reference': reference
    })

    print(f"Article {i+1} ({len(article.split())} words):")
    print(f"T5 Summary:  {summary}")
    print(f"Reference:   {reference[:150]}...")
    print(f"Compression: {len(article.split())} → {len(summary.split())} words")
    print("-" * 60)

print("\nT5 abstractive summarization successful")
print(f"Average input length:  {sum([r['article_length'] for r in t5_results])//5} words")
print(f"Average output length: {sum([r['summary_length'] for r in t5_results])//5} words")

=== T5 ABSTRACTIVE SUMMARIZATION ===

Loading T5 model... (this takes 3-4 minutes)


Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

T5 model loaded successfully
Model: t5-small (~60 million parameters)

Testing T5 on 5 real articles:

Article 1 (455 words):
T5 Summary:  the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. he will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II" despite his growing fame, he says he is keeping his feet firmly on the ground.
Reference:   Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's e...
Compression: 455 → 58 words
------------------------------------------------------------
Article 2 (698 words):
T5 Summary:  inmates with most severe mental illnesses are incarcerated until they're ready to appear in court. judge says arrests often result from confrontations with police. prisoners are wearing sleeveless robes, wearing sleeveless robes.
Reference:   Mentally ill inmates in Miami are housed 

In [ ]:
# Cell 6 — Model Comparison
print("=== MODEL COMPARISON — ALL THREE APPROACHES ===\n")

article = df['article'][0]
reference = df['highlights'][0]

textrank_sum = textrank_summarize(article, num_sentences=3)
bert_sum = bert_summarize(article, num_sentences=3)
t5_sum = t5_summarize(' '.join(article.split()[:800]))

print(f"ORIGINAL ARTICLE: {len(article.split())} words")
print(f"\nREFERENCE SUMMARY (human written):")
print(reference)

print(f"\n--- TextRank (Extractive) ---")
print(f"Length: {len(textrank_sum.split())} words")
print(textrank_sum[:300])

print(f"\n--- BERT (Extractive) ---")
print(f"Length: {len(bert_sum.split())} words")
print(bert_sum[:300])

print(f"\n--- T5 (Abstractive) ---")
print(f"Length: {len(t5_sum.split())} words")
print(t5_sum)

print("\n=== SUMMARY TABLE ===")
print(f"{'Model':<15} {'Type':<15} {'Output Length':<15}")
print("-" * 45)
print(f"{'TextRank':<15} {'Extractive':<15} {len(textrank_sum.split()):<15}")
print(f"{'BERT':<15} {'Extractive':<15} {len(bert_sum.split()):<15}")
print(f"{'T5':<15} {'Abstractive':<15} {len(t5_sum.split()):<15}")
print(f"{'Reference':<15} {'Human':<15} {len(reference.split()):<15}")
print("\nAll three models running successfully on real data")

=== MODEL COMPARISON — ALL THREE APPROACHES ===

ORIGINAL ARTICLE: 455 words

REFERENCE SUMMARY (human written):
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .

--- TextRank (Extractive) ---
Length: 97 words
Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. Despite his growing fame and riches, the actor says he

--- BERT (Extractive) ---
Length: 39 words
Details of how he'll mark his landmark birthday are under wraps. His agent and publicist had no comment on his plans. Despite his growing fame and riches, the actor says he is keeping his feet firmly on the ground.

--- T5 (Abstractive) ---
Length: 58 words
the young actor sa

In [ ]:
# Cell 7 — Save all model results to Google Drive
import pandas as pd
import json

print("=== SAVING ALL MODEL RESULTS ===\n")

# Run all 3 models on 50 articles and save results
print("Running all 3 models on 50 articles...")
print("(This takes about 5 minutes)")

all_results = []

for i in range(50):
    if i % 10 == 0:
        print(f"  Processing article {i}/50...")

    article = df['article'][i]
    reference = df['highlights'][i]
    article_short = ' '.join(article.split()[:800])

    tr_summary = textrank_summarize(article, num_sentences=3)
    bert_summary = bert_summarize(article, num_sentences=3)
    t5_summary = t5_summarize(article_short)

    all_results.append({
        'article_id': i,
        'article': article,
        'reference': reference,
        'textrank_summary': tr_summary,
        'bert_summary': bert_summary,
        't5_summary': t5_summary,
        'article_words': len(article.split()),
        'textrank_words': len(tr_summary.split()),
        'bert_words': len(bert_summary.split()),
        't5_words': len(t5_summary.split()),
        'reference_words': len(reference.split())
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv(
    '/content/drive/MyDrive/DocSnap_Project/model_results.csv',
    index=False
)

print(f"\nResults saved to Google Drive")
print(f"Total articles processed: {len(results_df)}")
print(f"\nAverage output lengths:")
print(f"  TextRank : {results_df['textrank_words'].mean():.0f} words")
print(f"  BERT     : {results_df['bert_words'].mean():.0f} words")
print(f"  T5       : {results_df['t5_words'].mean():.0f} words")
print(f"  Reference: {results_df['reference_words'].mean():.0f} words")
print("\nDay 3 complete — models saved and ready for ROUGE evaluation")

=== SAVING ALL MODEL RESULTS ===

Running all 3 models on 50 articles...
(This takes about 5 minutes)
  Processing article 0/50...
  Processing article 10/50...
  Processing article 20/50...
  Processing article 30/50...
  Processing article 40/50...

Results saved to Google Drive
Total articles processed: 50

Average output lengths:
  TextRank : 90 words
  BERT     : 61 words
  T5       : 43 words
  Reference: 42 words

Day 3 complete — models saved and ready for ROUGE evaluation


In [ ]:
# Cell 8 — Fine-tune T5 on CNN/DailyMail data
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch
import torch.optim as optim
import warnings
warnings.filterwarnings('ignore')

print("=== FINE-TUNING T5 ON CNN/DAILYMAIL ===\n")
print("This will take approximately 30-45 minutes on T4 GPU")

# Dataset class
class SummarizationDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=512, max_target=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        article = "summarize: " + str(self.data.iloc[idx]['article'])
        summary = str(self.data.iloc[idx]['highlights'])

        input_enc = self.tokenizer(
            article,
            max_length=self.max_input,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        target_enc = self.tokenizer(
            summary,
            max_length=self.max_target,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            'input_ids': input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels': labels
        }

# Use 500 samples for fine-tuning
train_df = df[:500].copy()
train_df = train_df.rename(columns={'highlights': 'highlights'})

print(f"Training samples: {len(train_df)}")

# Create dataset and dataloader
train_dataset = SummarizationDataset(train_df, t5_tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

# Fine-tune
t5_model.train()
optimizer = optim.AdamW(t5_model.parameters(), lr=5e-5)

print("\nStarting fine-tuning...")
print(f"Epochs: 3")
print(f"Batch size: 4")
print(f"Training samples: 500")
print(f"Learning rate: 5e-5\n")

for epoch in range(3):
    total_loss = 0
    batch_count = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels'].to('cuda')

        optimizer.zero_grad()

        outputs = t5_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        if batch_idx % 20 == 0:
            print(f"  Epoch {epoch+1}/3 | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / batch_count
    print(f"\nEpoch {epoch+1} complete — Average Loss: {avg_loss:.4f}\n")

# Save fine-tuned model
print("Saving fine-tuned T5 model to Google Drive...")
t5_model.save_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_finetuned')
t5_tokenizer.save_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_finetuned')

print("Fine-tuned T5 model saved successfully")
print("Location: /content/drive/MyDrive/DocSnap_Project/t5_finetuned")

=== FINE-TUNING T5 ON CNN/DAILYMAIL ===

This will take approximately 30-45 minutes on T4 GPU
Training samples: 500

Starting fine-tuning...
Epochs: 3
Batch size: 4
Training samples: 500
Learning rate: 5e-5

  Epoch 1/3 | Batch 0/125 | Loss: 2.8979
  Epoch 1/3 | Batch 20/125 | Loss: 2.5401
  Epoch 1/3 | Batch 40/125 | Loss: 2.4347
  Epoch 1/3 | Batch 60/125 | Loss: 2.6894
  Epoch 1/3 | Batch 80/125 | Loss: 2.6876
  Epoch 1/3 | Batch 100/125 | Loss: 2.3871
  Epoch 1/3 | Batch 120/125 | Loss: 2.2803

Epoch 1 complete — Average Loss: 2.2887

  Epoch 2/3 | Batch 0/125 | Loss: 1.7069
  Epoch 2/3 | Batch 20/125 | Loss: 1.9808
  Epoch 2/3 | Batch 40/125 | Loss: 2.1920
  Epoch 2/3 | Batch 60/125 | Loss: 2.2742
  Epoch 2/3 | Batch 80/125 | Loss: 1.6429
  Epoch 2/3 | Batch 100/125 | Loss: 2.3285
  Epoch 2/3 | Batch 120/125 | Loss: 1.5408

Epoch 2 complete — Average Loss: 2.1333

  Epoch 3/3 | Batch 0/125 | Loss: 2.3955
  Epoch 3/3 | Batch 20/125 | Loss: 2.1404
  Epoch 3/3 | Batch 40/125 | Loss: 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned T5 model saved successfully
Location: /content/drive/MyDrive/DocSnap_Project/t5_finetuned


In [ ]:
# Cell 9 — Fine-tune BERT for Extractive Summarization
from transformers import BertTokenizer, BertForSequenceClassification
import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from nltk.tokenize import sent_tokenize
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("=== FINE-TUNING BERT FOR EXTRACTIVE SUMMARIZATION ===\n")

class BertExtractiveSumDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=128):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.sentences = []
        self.labels = []

        print("Preparing BERT training data...")
        for idx, row in dataframe.iterrows():
            article = row['article']
            reference = row['highlights']

            sentences = sent_tokenize(article)[:10]
            ref_words = set(reference.lower().split())

            for sent in sentences:
                sent_words = set(sent.lower().split())
                overlap = len(sent_words & ref_words) / (len(sent_words) + 1)
                label = 1 if overlap > 0.1 else 0
                self.sentences.append(sent)
                self.labels.append(label)

        print(f"Total sentences prepared: {len(self.sentences)}")
        pos = sum(self.labels)
        print(f"Positive (summary) sentences: {pos}")
        print(f"Negative sentences: {len(self.labels) - pos}")

    def __len__(self):
        return len(self.sentences)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.sentences[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Load BERT for classification
print("Loading BERT for sequence classification...")
bert_ft_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=2
)
bert_ft_model = bert_ft_model.cuda()

# Prepare dataset — use 200 articles
bert_train_df = df[:200].copy()
bert_dataset = BertExtractiveSumDataset(bert_train_df, tokenizer)
bert_loader = DataLoader(bert_dataset, batch_size=16, shuffle=True)

# Fine-tune
optimizer = optim.AdamW(bert_ft_model.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

print(f"\nStarting BERT fine-tuning...")
print(f"Epochs: 3 | Batch size: 16 | Learning rate: 2e-5\n")

bert_ft_model.train()
for epoch in range(3):
    total_loss = 0
    correct = 0
    total = 0

    for batch_idx, batch in enumerate(bert_loader):
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['label'].to('cuda')

        optimizer.zero_grad()
        outputs = bert_ft_model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = criterion(outputs.logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if batch_idx % 10 == 0:
            acc = correct / total * 100
            print(f"  Epoch {epoch+1}/3 | Batch {batch_idx}/{len(bert_loader)} | Loss: {loss.item():.4f} | Acc: {acc:.1f}%")

    avg_loss = total_loss / len(bert_loader)
    accuracy = correct / total * 100
    print(f"\nEpoch {epoch+1} complete — Loss: {avg_loss:.4f} | Accuracy: {accuracy:.1f}%\n")

# Save
print("Saving fine-tuned BERT model...")
bert_ft_model.save_pretrained('/content/drive/MyDrive/DocSnap_Project/bert_finetuned')
tokenizer.save_pretrained('/content/drive/MyDrive/DocSnap_Project/bert_finetuned')

print("Fine-tuned BERT saved successfully")
print("Location: /content/drive/MyDrive/DocSnap_Project/bert_finetuned")

=== FINE-TUNING BERT FOR EXTRACTIVE SUMMARIZATION ===

Loading BERT for sequence classification...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Preparing BERT training data...
Total sentences prepared: 1988
Positive (summary) sentences: 1535
Negative sentences: 453

Starting BERT fine-tuning...
Epochs: 3 | Batch size: 16 | Learning rate: 2e-5

  Epoch 1/3 | Batch 0/125 | Loss: 0.6240 | Acc: 68.8%
  Epoch 1/3 | Batch 10/125 | Loss: 0.6342 | Acc: 82.4%
  Epoch 1/3 | Batch 20/125 | Loss: 0.6132 | Acc: 78.0%
  Epoch 1/3 | Batch 30/125 | Loss: 0.4105 | Acc: 78.6%
  Epoch 1/3 | Batch 40/125 | Loss: 0.6560 | Acc: 78.2%
  Epoch 1/3 | Batch 50/125 | Loss: 0.4305 | Acc: 78.2%
  Epoch 1/3 | Batch 60/125 | Loss: 0.8073 | Acc: 76.5%
  Epoch 1/3 | Batch 70/125 | Loss: 0.5300 | Acc: 76.6%
  Epoch 1/3 | Batch 80/125 | Loss: 0.3071 | Acc: 77.5%
  Epoch 1/3 | Batch 90/125 | Loss: 0.6142 | Acc: 76.8%
  Epoch 1/3 | Batch 100/125 | Loss: 0.4266 | Acc: 77.4%
  Epoch 1/3 | Batch 110/125 | Loss: 0.5536 | Acc: 77.0%
  Epoch 1/3 | Batch 120/125 | Loss: 0.2136 | Acc: 77.4%

Epoch 1 complete — Loss: 0.5293 | Accuracy: 77.2%

  Epoch 2/3 | Batch 0/125 | L

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned BERT saved successfully
Location: /content/drive/MyDrive/DocSnap_Project/bert_finetuned


In [ ]:
# Cell 10 — Load and Fine-tune T5-base
from transformers import T5Tokenizer, T5ForConditionalGeneration
from torch.utils.data import Dataset, DataLoader
import torch
import torch.optim as optim
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("=== FINE-TUNING T5-BASE ===\n")
print("Loading t5-base (220M parameters)...")
print("This is larger than t5-small (60M) — better quality expected\n")

# Load t5-base
t5_base_tokenizer = T5Tokenizer.from_pretrained('t5-base')
t5_base_model = T5ForConditionalGeneration.from_pretrained('t5-base')
t5_base_model = t5_base_model.cuda()
t5_base_model.train()

print("T5-base loaded successfully")
print("Parameters: ~220 million")

# Load data from Drive
df = pd.read_csv('/content/drive/MyDrive/DocSnap_Project/docsnap_data.csv')
print(f"Training data loaded: {len(df)} articles")

# Dataset class
class SumDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_input=512, max_target=128):
        self.data = dataframe
        self.tokenizer = tokenizer
        self.max_input = max_input
        self.max_target = max_target

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        article = "summarize: " + str(self.data.iloc[idx]['article'])
        summary = str(self.data.iloc[idx]['highlights'])
        input_enc = self.tokenizer(
            article, max_length=self.max_input,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        target_enc = self.tokenizer(
            summary, max_length=self.max_target,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100
        return {
            'input_ids': input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels': labels
        }

# Use 500 samples
train_df = df[:500].copy()
train_dataset = SumDataset(train_df, t5_base_tokenizer)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

optimizer = optim.AdamW(t5_base_model.parameters(), lr=5e-5)

print(f"\nStarting T5-base fine-tuning...")
print(f"Epochs: 3 | Batch size: 4 | Samples: 500 | LR: 5e-5\n")

for epoch in range(3):
    total_loss = 0
    batch_count = 0

    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels = batch['labels'].to('cuda')

        optimizer.zero_grad()
        outputs = t5_base_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

        if batch_idx % 20 == 0:
            print(f"  Epoch {epoch+1}/3 | Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / batch_count
    print(f"\nEpoch {epoch+1} complete — Average Loss: {avg_loss:.4f}\n")

# Save
print("Saving T5-base model to Google Drive...")
t5_base_model.save_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')
t5_base_tokenizer.save_pretrained('/content/drive/MyDrive/DocSnap_Project/t5_base_finetuned')

print("T5-base fine-tuned model saved successfully")
print("Location: /content/drive/MyDrive/DocSnap_Project/t5_base_finetuned")

=== FINE-TUNING T5-BASE ===

Loading t5-base (220M parameters)...
This is larger than t5-small (60M) — better quality expected



spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5-base loaded successfully
Parameters: ~220 million
Training data loaded: 5000 articles

Starting T5-base fine-tuning...
Epochs: 3 | Batch size: 4 | Samples: 500 | LR: 5e-5

  Epoch 1/3 | Batch 0/125 | Loss: 2.5466
  Epoch 1/3 | Batch 20/125 | Loss: 2.5430
  Epoch 1/3 | Batch 40/125 | Loss: 1.8227
  Epoch 1/3 | Batch 60/125 | Loss: 2.0476
  Epoch 1/3 | Batch 80/125 | Loss: 1.5880
  Epoch 1/3 | Batch 100/125 | Loss: 1.7467
  Epoch 1/3 | Batch 120/125 | Loss: 1.7770

Epoch 1 complete — Average Loss: 1.7193

  Epoch 2/3 | Batch 0/125 | Loss: 1.2654
  Epoch 2/3 | Batch 20/125 | Loss: 1.3677
  Epoch 2/3 | Batch 40/125 | Loss: 1.4322
  Epoch 2/3 | Batch 60/125 | Loss: 1.3720
  Epoch 2/3 | Batch 80/125 | Loss: 0.9632
  Epoch 2/3 | Batch 100/125 | Loss: 1.6308
  Epoch 2/3 | Batch 120/125 | Loss: 1.9660

Epoch 2 complete — Average Loss: 1.4822

  Epoch 3/3 | Batch 0/125 | Loss: 0.9965
  Epoch 3/3 | Batch 20/125 | Loss: 0.9611
  Epoch 3/3 | Batch 40/125 | Loss: 0.9560
  Epoch 3/3 | Batch 60/125

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T5-base fine-tuned model saved successfully
Location: /content/drive/MyDrive/DocSnap_Project/t5_base_finetuned
